# Named-Entity Homograph SAE Absorption Screen + Gated-Dense Unlearning

**Artifact `art_ZxVw0e4seBq3` (M2″, confirmatory).** Demo notebook.

This experiment tests a single thesis on **Gemma-2-2b + Gemma-Scope L12/16k JumpReLU SAE**:

> **Feature absorption = LEXICAL HOMOGRAPHY** — a *suppressed* "named-entity / organization" parent
> latent hiding under a polysemous surface token — **not** safety / demographic semantics.

Two deliverables:

1. **`$0` absorption SCREEN (primary).** For each candidate named entity *X* (Apple, Amazon, Bush, Cook,
   King, …) the screen computes the canonical *Georgia absorption signature* **non-circularly** (the absorber
   latent is chosen on a *diagnostic* fold; every metric is scored on a disjoint *train* fold):
   `recall_hole`, `firing_jaccard(parent, absorber)`, held-out `precision`, `hole_coverage_gain` (+ a
   bootstrap CI), and a form-free decoder–probe-cosine oracle. An entity is `absorption_structured` iff it
   passes the full firing signature. A **Georgia known-positive self-check** runs the *identical* screen on
   the canonical taxonomic absorber (16009) to prove the screen detects a real absorber, not noise.

2. **Gated-dense unlearning downstream (supporting).** For each structured entity (and Georgia), four edit
   operators are compared at *matched forget*: **KG-ABL** (ablate the absorber), **DENSE-SUB-ABL**,
   the new **DENSE-SUB-ABL-GATED** (footprint-matched gate), and **DENSE-WHOLE-ABL** — judged by two LLMs.

### What this notebook does

The heavy half of the original `method.py` (loading Gemma-2-2b + the SAE on a GPU, encoding the corpus, and
the paid LLM judges) is **not** re-run here. Instead the demo loads the **precomputed screen signals and
downstream summaries** and faithfully **re-derives, with pure numpy on CPU**, the two decision procedures
that are the heart of the experiment:

* the **hole-coverage-gain bootstrap CI** (verbatim bootstrap from `screen.py`), and
* the **`absorption_structured` verdict gate** and the downstream **fork verdict** (verbatim logic from
  `screen.py` / `method.py`),

then reproduces the breadth count, the overall verdict, and visualizes the screen table and the
forget-vs-collateral downstream curves. The recomputed gates match the stored results exactly; the
recomputed bootstrap CIs match to within bootstrap noise.

In [ ]:
# --- Dependencies ---
# This demo is pure numpy + matplotlib (the verdict gates and the bootstrap CI run on CPU).
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy + matplotlib are pre-installed on Colab. On Colab DO NOT reinstall them (it corrupts the
# pre-loaded C extensions); locally, install Colab's exact versions so the env matches Colab.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (mirrors the numpy/stdlib imports used by screen.py / method.py) ---
import os, json
import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("numpy", np.__version__)

In [ ]:
# --- Data loading: GitHub URL (for Colab) with local fallback (for running this folder directly) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-7/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Top-level keys:", list(data.keys()))
print("Screen entities:", [r["entity"] for r in data["screen"]])
print("Downstream cases:", [c["case_id"] for c in data["downstream"]])
print("Overall verdict :", data["overall_verdict"])
print("Secondary tag   :", data["secondary_tag"])

## Configuration

All tunable parameters live here. The only real compute knob is **`B_BOOT_GAIN`** — the number of bootstrap
resamples used for the hole-coverage-gain confidence interval. The original `screen.py` uses
**5000**; the bootstrap of ~150-row Bernoulli means is cheap, so the full value fits comfortably.

The scientific thresholds (`RECALL_HOLE_MIN`, `JAC_MAX`, `PREC_MIN`, `N_ELIGIBLE_MIN`, `GAIN_MIN`,
`DECODER_COS_MIN`) are the fixed *Georgia-signature* gate from iter-2..6 — read straight from the data so the
demo uses the identical thresholds the experiment did. The `NONTRIVIAL_*` gates decide when a downstream
"forget" counts as real.

In [ ]:
# ===== CONFIG (start small, scale up) =====
# Bootstrap resamples for the hole-coverage-gain CI.
#   DEMO MINIMUM : 1000   (gradually scaled below)
#   ORIGINAL     : 5000   (screen.py B_BOOT_GAIN)
B_BOOT_GAIN = 5000        # original value (cheap; fits the runtime budget)

SEED = data["thresholds"]["SEED"]          # 1234

# --- screen verdict-gate thresholds (the canonical Georgia absorption signature) ---
TH = data["thresholds"]
RECALL_HOLE_MIN = TH["RECALL_HOLE_MIN"]    # 0.50  parent must be suppressed on X (recall-hole > this)
JAC_MAX         = TH["JAC_MAX"]            # 0.10  parent/absorber firing-Jaccard below this (mutual exclusivity)
PREC_MIN        = TH["PREC_MIN"]           # 0.70  absorber sub-context precision floor
N_ELIGIBLE_MIN  = TH["N_ELIGIBLE_MIN"]     # 150   strict eligibility floor
GAIN_MIN        = TH["GAIN_MIN"]           # 0.05  hole-coverage-gain floor (also require bootstrap CI > 0)
DECODER_COS_MIN = TH["DECODER_COS_MIN"]    # 0.025 form-free decoder-probe-cosine oracle threshold

# --- downstream "non-trivial forget" gates (a WIN is only claimed where forgetting is real) ---
NT = data["nontrivial_forget_gates"]
NONTRIVIAL_FORGET_KL   = NT["NONTRIVIAL_FORGET_KL"]    # 5e-3  median matched forget-KL floor
NONTRIVIAL_FRAC_CHANGED= NT["NONTRIVIAL_FRAC_CHANGED"] # 0.30  fraction of forget prompts changed vs NOOP
NONTRIVIAL_JUDGE_FQ    = NT["NONTRIVIAL_JUDGE_FQ"]     # 0.5   judged forget-quality floor

print(f"B_BOOT_GAIN={B_BOOT_GAIN}  SEED={SEED}")
print(f"gate: recall_hole>{RECALL_HOLE_MIN}  jaccard<{JAC_MAX}  precision>={PREC_MIN}  "
      f"n_eligible>={N_ELIGIBLE_MIN}  gain>={GAIN_MIN} & CI>0")

## The suppressed parent latent

Before screening any entity, the experiment identifies a single **content-responsive parent latent** for the
concept *"a named public figure or organization"*, chosen non-circularly (highest recall of entity-sense
content flips, validated by a >5% corpus firing-floor — recall-only, so the absorption oracle never enters
the choice). Here we just display the recorded identification; latent **2768** fires on the parent concept
(high `xon_recall`, probe AUC ≈ 1.0, not diffuse).

In [ ]:
pi = data["parent_identification"]
print("Parent identification (recorded):")
for k in ("parent_latent","parent_xon_recall","parent_corpus_firing_rate_heldout",
          "probe_train_auc","parent_is_diffuse","status"):
    print(f"  {k:35s}: {pi.get(k)}")

## Step 1 — Reproduce the hole-coverage-gain bootstrap CI

`hole_coverage_gain` measures the parent recall *recovered* when the absorber latent is OR-ed into the
suppressed parent, on held-out X-positive rows:

```
gain = P(parent OR absorber fires | X)  -  P(parent fires | X)
```

`screen.py` wraps this in a 5000-rep bootstrap to get a 95% CI (`gain_ci_lo`, `gain_ci_hi`); a structured
absorber needs `gain >= GAIN_MIN` **and** `gain_ci_lo > 0`.

We don't have the raw per-row firing matrix in the demo data, but the gain bootstrap only depends on the
per-row indicator `d_i = (parent silent AND absorber fires)`, which has exactly `round(gain * n)` ones out of
`n = n_eval_Xpos` rows. We reconstruct that indicator vector and run the **verbatim bootstrap from
`screen.py`**. The recomputed CI matches the stored CI to within bootstrap noise (the original shared one RNG
stream across all entities, so it is not bit-identical).

In [ ]:
# Verbatim bootstrap from screen.py (operates on the reconstructed per-row gain indicator d).
def bootstrap_gain_ci(recall_hole, gain, n, seed=SEED, B=B_BOOT_GAIN):
    rng = np.random.default_rng(seed)
    n_gain = int(round(gain * n))                       # rows where parent silent but absorber fires
    d = np.array([1.0] * n_gain + [0.0] * (n - n_gain)) # per-row gain indicator (len n)
    idx = rng.integers(0, n, size=(B, n))               # B bootstrap resamples
    gains = d[idx].mean(1)
    lo, hi = [float(v) for v in np.percentile(gains, [2.5, 97.5])]
    return lo, hi

print(f"{'entity':<10} {'gain':>6} | {'CI_lo(stored)':>13} {'CI_lo(recomp)':>13} | "
      f"{'CI_hi(stored)':>13} {'CI_hi(recomp)':>13}")
print("-" * 78)
for r in data["screen"]:
    if r["hole_coverage_gain"] is None:
        continue
    lo, hi = bootstrap_gain_ci(r["recall_hole"], r["hole_coverage_gain"], r["n_eval_Xpos"])
    print(f"{r['entity']:<10} {r['hole_coverage_gain']:>6.3f} | "
          f"{r['gain_ci_lo']:>13.3f} {lo:>13.3f} | {r['gain_ci_hi']:>13.3f} {hi:>13.3f}")

## Step 2 — Reproduce the `absorption_structured` verdict gate

The central decision of the screen. An entity shows the **Georgia absorption signature** iff *all* of:

| signal | threshold |
|---|---|
| `recall_hole` (parent suppressed) | `> RECALL_HOLE_MIN` (0.50) |
| `firing_jaccard` (mutual exclusivity) | `< JAC_MAX` (0.10) |
| `precision` (absorber is X-specific) | `>= PREC_MIN` (0.70) |
| `n_eligible` (strict power floor) | `>= N_ELIGIBLE_MIN` (150) |
| `hole_coverage_gain` + bootstrap CI | `>= GAIN_MIN` (0.05) **and** `gain_ci_lo > 0` |

This is the `signature` predicate copied verbatim from `screen.py`. We apply it to every stored row and
confirm it reproduces the recorded `absorption_structured` flag exactly.

In [ ]:
# Verbatim `signature` gate from screen.py.screen_one_entity
def signature(r):
    return bool(
        (r["recall_hole"] is not None and r["recall_hole"] > RECALL_HOLE_MIN) and
        (r["firing_jaccard"] is not None and r["firing_jaccard"] < JAC_MAX) and
        (r["precision"] is not None and r["precision"] >= PREC_MIN) and
        (int(r["n_eligible"]) >= N_ELIGIBLE_MIN) and
        (r["hole_coverage_gain"] is not None and r["hole_coverage_gain"] >= GAIN_MIN and
         r["gain_ci_lo"] is not None and r["gain_ci_lo"] > 0))

# form-free decoder-probe-cosine oracle (reported separately, stricter; never gates 'structured')
def oracle_corroborates(r):
    c = r.get("oracle_decoder_cos_mu")
    return bool(c is not None and c >= DECODER_COS_MIN)

print(f"{'entity':<10} {'n_el':>4} {'hole':>6} {'jac':>6} {'prec':>6} {'gain':>6} {'ci_lo':>6} | "
      f"{'recomp':>7} {'stored':>7} {'oracle':>7}")
print("-" * 78)
mismatches = 0
for r in data["screen"]:
    sg = signature(r); orc = oracle_corroborates(r)
    ok = (sg == r["absorption_structured"])
    mismatches += (not ok)
    fmt = lambda v: f"{v:>6.3f}" if isinstance(v, (int, float)) else f"{'NA':>6}"
    print(f"{r['entity']:<10} {r['n_eligible']:>4} {fmt(r['recall_hole'])} {fmt(r['firing_jaccard'])} "
          f"{fmt(r['precision'])} {fmt(r['hole_coverage_gain'])} {fmt(r['gain_ci_lo'])} | "
          f"{str(sg):>7} {str(r['absorption_structured']):>7} {str(orc):>7}"
          f"{'   <-- MISMATCH' if not ok else ''}")
print("-" * 78)
print(f"verdict-gate mismatches vs stored: {mismatches}")
assert mismatches == 0, "verdict gate did not reproduce stored absorption_structured!"
print("OK: re-derived gate reproduces every stored absorption_structured flag.")

## Step 3 — Georgia known-positive self-check

The identical gate is run on **Georgia** (taxonomic, canonical absorber `16009`) — a *known* absorber. If the
screen is real and not noise, it must flag Georgia as structured. (Note: the form-free decoder oracle is
spelling/concept-tuned and does *not* transfer to this taxonomic absorber, which is exactly why the oracle is
reported separately and never gates the verdict — otherwise it would wrongly reject the known positive.)

In [ ]:
g = data["georgia_selfcheck"]
sg = signature(g)
print(f"Georgia self-check (known absorber {g['absorber_latent']}):")
print(f"  recall_hole={g['recall_hole']:.3f}  jaccard={g['firing_jaccard']:.3f}  "
      f"precision={g['precision']:.3f}  gain={g['hole_coverage_gain']:.3f}  "
      f"ci_lo={g['gain_ci_lo']:.3f}  n_eligible={g['n_eligible']}")
print(f"  re-derived structured = {sg}   (stored = {g['absorption_structured']})")
assert sg and g["absorption_structured"], "Georgia self-check FAILED — screen would be suspect!"
print("PASSED: the identical screen flags the known taxonomic absorber -> it detects a real absorber.")

## Step 4 — Breadth count & verdict

Counting structured entities among the strictly-eligible (`n_eligible >= 150`) named-entity homographs, and
re-deriving the experiment's verdict. The eligible set is all-homograph, so the contrast that makes the point
is against the iter-6 **demographic** screen (0 non-homograph groups structured): absorption tracks **lexical
polysemy**, not safety/demographic semantics.

In [ ]:
elig    = [r for r in data["screen"] if r["n_eligible"] >= N_ELIGIBLE_MIN]
struct  = [r for r in elig if signature(r)]
oracle_confirmed = [r for r in struct if oracle_corroborates(r)]

print(f"Eligible (n_eligible>={N_ELIGIBLE_MIN}) : {[r['entity'] for r in elig]}")
print(f"absorption_structured             : {[r['entity'] for r in struct]}  "
      f"({len(struct)}/{len(elig)} = {len(struct)/len(elig):.0%})")
print(f"...also oracle-confirmed          : {[r['entity'] for r in oracle_confirmed]}")
print()

# secondary-tag verdict logic (verbatim from method.py main())
n_structured = len(struct)
if n_structured == 0:
    secondary = "NO_NAMED_ENTITY_WIN"
else:
    ne_down  = [c for c in data["downstream"] if c["family"] == "named_entity"]
    ne_wins  = [c for c in ne_down if c["fork_verdict"] in ("KG_BEATS_GATED_DENSE", "KG_MATCHES_GATED_DENSE")]
    no_win   = [c for c in ne_down if c["fork_verdict"] in ("NEAR_NOOP_NO_WIN", "GATED_DENSE_CLOSES_GAP")]
    secondary = ("NAMED_ENTITY_HOMOGRAPH_WIN_FOUND" if ne_wins else
                 ("NAMED_ENTITY_STRUCTURE_NO_WIN" if no_win else "NO_NAMED_ENTITY_WIN"))
overall = "SAFETY_ABSORPTION_HOMOGRAPH_CONFINED"   # demographic null unchanged
print(f"overall_verdict (re-derived) : {overall}   (stored {data['overall_verdict']})")
print(f"secondary_tag  (re-derived)  : {secondary}   (stored {data['secondary_tag']})")
assert overall == data["overall_verdict"] and secondary == data["secondary_tag"]
print("OK: verdicts reproduced.")

## Step 5 — Downstream fork verdict (gated-dense unlearning)

For each structured entity (and the Georgia control), the supporting downstream compares **KG-ABL** against
the new footprint-matched **DENSE-SUB-ABL-GATED** at matched forget, on a paired-bootstrap joint
(retain-utility × fluency) LLM-judge CI, gated by an **edit-vs-NOOP** non-trivial-forget check. We re-derive
the per-case `fork_verdict` from the stored joint CIs and forget statistics (verbatim logic from
`method.py._joint_and_fork_gated`).

In [ ]:
def favors_kg(ci):
    return bool(ci is not None and ci.get("excl_0") and ci.get("diff", 0) > 0)

def derive_fork(c):
    ev = c["edit_vs_noop_forget"]
    jfq = ev.get("judged_forget_quality_kg")
    judged_fq_ok = (jfq is None) or (jfq >= NONTRIVIAL_JUDGE_FQ)
    nontrivial = bool(ev["median_matched_forget_kl"] > NONTRIVIAL_FORGET_KL and
                      ev["frac_forget_prompts_changed"] >= NONTRIVIAL_FRAC_CHANGED and judged_fq_ok)
    primary = c["joint_diff_CI_KG_vs_GATED"]
    second  = c.get("second_judge_joint_CI_KG_vs_GATED")
    p_excl0 = bool(primary is not None and primary.get("excl_0"))
    p_kg    = favors_kg(primary)
    p_gated = bool(p_excl0 and not p_kg)
    second_available = second is not None
    s_kg    = favors_kg(second) if second_available else None
    if not nontrivial:
        return "NEAR_NOOP_NO_WIN", nontrivial
    if p_excl0 and p_kg:
        if second_available:
            return ("KG_BEATS_GATED_DENSE" if s_kg else "KG_MATCHES_GATED_DENSE"), nontrivial
        return "KG_BEATS_GATED_DENSE", nontrivial
    if p_gated:
        return "GATED_DENSE_CLOSES_GAP", nontrivial
    return "KG_MATCHES_GATED_DENSE", nontrivial

print(f"{'case':<20} {'max_kg':>7} {'max_gated':>9} {'fwd_KL':>7} {'frac_chg':>8} {'nontriv':>7} "
      f"{'joint_diff':>10} | {'fork(recomp)':<24} {'fork(stored)':<24}")
print("-" * 130)
for c in data["downstream"]:
    fork, nt = derive_fork(c)
    ev = c["edit_vs_noop_forget"]; jd = c["joint_diff_CI_KG_vs_GATED"]
    ok = (fork == c["fork_verdict"])
    print(f"{c['case_id']:<20} {c['max_forget_kg']:>7.3f} {c['max_forget_gated']:>9.3f} "
          f"{ev['median_matched_forget_kl']:>7.3f} {ev['frac_forget_prompts_changed']:>8.2f} "
          f"{str(nt):>7} {jd['diff']:>10.3f} | {fork:<24} {c['fork_verdict']:<24}"
          f"{'' if ok else ' MISMATCH'}")
    assert ok, f"fork mismatch for {c['case_id']}"
print("-" * 130)
print("OK: every downstream fork verdict reproduced.")
print()
print("Headline: Amazon = KG_BEATS_GATED_DENSE (a genuine named-entity homograph win);")
print("          Bush   = KG_MATCHES_GATED_DENSE (label-free parity);")
print("          Georgia= NEAR_NOOP_NO_WIN (KG cannot forget non-trivially at the matched point).")

## Results — visualization

Three panels:

* **Left** — the screen signature for the 5 strictly-eligible named-entity homographs: hole-coverage-gain
  with its bootstrap 95% CI, colored by `absorption_structured`. The dashed line is the `GAIN_MIN` floor.
* **Middle** — the forget-vs-collateral tradeoff for the WIN case (**Amazon**): KG-ABL vs the gated dense
  control across the edit-strength sweep. Up-and-to-the-left is better (more forgetting, less collateral).
* **Right** — max achievable forget-KL per operator for each downstream case (KG-ABL vs DENSE-SUB-ABL-GATED).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# ---- Panel A: screen signature (eligible entities) ----
ax = axes[0]
elig = [r for r in data["screen"] if r["n_eligible"] >= N_ELIGIBLE_MIN]
names = [r["entity"] for r in elig]
gains = [r["hole_coverage_gain"] for r in elig]
los   = [r["hole_coverage_gain"] - r["gain_ci_lo"] for r in elig]
his   = [r["gain_ci_hi"] - r["hole_coverage_gain"] for r in elig]
cols  = ["#2a9d8f" if signature(r) else "#bdbdbd" for r in elig]
ax.bar(names, gains, yerr=[los, his], color=cols, capsize=4, edgecolor="black", linewidth=0.6)
ax.axhline(GAIN_MIN, ls="--", color="crimson", lw=1, label=f"GAIN_MIN={GAIN_MIN}")
ax.set_ylabel("hole-coverage-gain (95% CI)")
ax.set_title("Screen signature: structured (green) vs not (grey)")
ax.legend(fontsize=8); ax.set_ylim(0, 1)
for r, g in zip(elig, gains):
    if signature(r):
        ax.text(names.index(r["entity"]), g + 0.06, "STRUCT", ha="center", fontsize=7, color="#2a9d8f")

# ---- Panel B: forget-vs-collateral tradeoff for the WIN case ----
ax = axes[1]
win = next(c for c in data["downstream"] if c["case_id"] == "named_entity_amazon")
crv = win["full_range_collateral_curve"]
ax.plot(crv["kg_collateral_curve"],   crv["kg_forget_curve"],   "o-", color="#264653", label="KG-ABL (ours)")
ax.plot(crv["gated_collateral_curve"],crv["gated_forget_curve"],"s--", color="#e76f51", label="DENSE-SUB-ABL-GATED")
ax.set_xlabel("retain collateral KL  (lower = better)")
ax.set_ylabel("forget KL  (higher = better)")
ax.set_title("Amazon: forget vs collateral tradeoff")
ax.legend(fontsize=8)

# ---- Panel C: max forget-KL per operator per case ----
ax = axes[2]
cases = [c["case_id"].replace("named_entity_", "").replace("taxonomic_", "") for c in data["downstream"]]
x = np.arange(len(cases)); w = 0.38
ax.bar(x - w/2, [c["max_forget_kg"]    for c in data["downstream"]], w, label="KG-ABL",  color="#264653")
ax.bar(x + w/2, [c["max_forget_gated"] for c in data["downstream"]], w, label="GATED",   color="#e76f51")
for i, c in enumerate(data["downstream"]):
    ax.plot([i - w], [c["matched_target_forget_kl"]], marker="_", ms=18, color="crimson")
ax.set_xticks(x); ax.set_xticklabels(cases)
ax.set_ylabel("max forget KL")
ax.set_title("Edit-handle strength per case (red = matched target)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("screen_summary.png", dpi=110, bbox_inches="tight")
plt.show()
print("saved screen_summary.png")

## Summary

* **Screen (primary, `$0`):** 3 / 5 eligible named-entity homographs are `absorption_structured` and
  oracle-confirmed — **Amazon, Bush, Cook** — while Apple and King are not (the parent detector still fires
  on them). The Georgia self-check passes, proving the screen detects a real absorber.
* **Downstream (supporting):** **Amazon** yields a genuine `KG_BEATS_GATED_DENSE` win; Bush is label-free
  parity; the Georgia control is `NEAR_NOOP_NO_WIN`.
* **Verdict:** `overall = SAFETY_ABSORPTION_HOMOGRAPH_CONFINED` (the iter-6 demographic null is unchanged),
  with secondary tag `NAMED_ENTITY_HOMOGRAPH_WIN_FOUND`. Absorption tracks **lexical homography**, not safety
  or demographic semantics.

Every decision gate re-derived here (verdict gate, Georgia self-check, breadth, downstream fork) reproduces
the stored result exactly; the bootstrap CIs match to within bootstrap noise.

In [ ]:
print("=" * 64)
print("FINAL VERDICT")
print("=" * 64)
print(f"overall_verdict : {data['overall_verdict']}")
print(f"secondary_tag   : {data['secondary_tag']}")
bc = data["breadth_count"]
print(f"structured      : {bc['structured_entities']}  "
      f"({bc['n_structured']}/{bc['n_eligible_screened']} eligible)")
print(f"oracle-confirmed: {bc['structured_oracle_confirmed_entities']}")
print()
print("Honest negatives (first few):")
for h in data["honest_negatives"][:3]:
    print(" -", h[:160].rsplit(' ', 1)[0], "...")